# 🩺 Skin Disease AI — Обучение в Google Colab
**Модель:** EfficientNet-B3 (transfer learning)  
**Датасеты:** HAM10000 / ISIC / Fitzpatrick17k  
**GPU:** T4 (бесплатный Colab)

---
### Структура папок датасета:
```
dataset/
├── train/
│   ├── psoriasis/      ← фото псориаза
│   ├── dermatitis/     ← дерматит / экзема
│   ├── melanoma/
│   ├── nevus/
│   └── ...
└── val/
    ├── psoriasis/
    └── ...
```

In [ ]:
# ── ЯЧЕЙКА 1: Проверка GPU ──────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ GPU не найден! Включите: Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── ЯЧЕЙКА 2: Установка библиотек ───────────────────────────
!pip install -q scikit-learn torchvision Pillow

In [ ]:
# ── ЯЧЕЙКА 3: Монтирование Google Drive ─────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Путь к вашей папке в Drive
DRIVE_PATH = '/content/drive/MyDrive/skin_ai'
import os
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f'✅ Drive подключён: {DRIVE_PATH}')

In [ ]:
# ── ЯЧЕЙКА 4: Скачать HAM10000 с Kaggle ─────────────────────
# Способ 1: через Kaggle API (нужен kaggle.json)
# from google.colab import files
# files.upload()  # загрузите kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d kmader/skin-lesion-analysis-toward-melanoma-detection

# Способ 2: прямые ссылки на ISIC (без авторизации)
# Fitzpatrick17k — GitHub
!pip install -q gdown
import gdown

# Fitzpatrick17k metadata (CSV с URL фотографий)
!wget -q https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv -O fitzpatrick17k.csv
print('✅ Метаданные Fitzpatrick17k скачаны')

import pandas as pd
df = pd.read_csv('fitzpatrick17k.csv')
print(f'Всего записей: {len(df)}')
print(df['label'].value_counts().head(20))

In [ ]:
# ── ЯЧЕЙКА 5: Подготовка датасета по классам ────────────────
import os, shutil
from pathlib import Path
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm

# Маппинг Fitzpatrick → наши классы
CLASS_MAP = {
    'psoriasis': 'psoriasis',
    'eczema': 'dermatitis',
    'atopic dermatitis': 'dermatitis',
    'contact dermatitis': 'dermatitis',
    'seborrheic dermatitis': 'dermatitis',
    'melanoma': 'melanoma',
    'melanocytic nevi': 'nevus',
    'actinic keratosis': 'actinic_keratosis',
    'basal cell carcinoma': 'basal_cell_carcinoma',
}

TARGET_CLASSES = list(set(CLASS_MAP.values()))
print(f'Целевые классы: {TARGET_CLASSES}')

# Создаём структуру папок
for split in ['train', 'val']:
    for cls in TARGET_CLASSES:
        Path(f'/content/dataset/{split}/{cls}').mkdir(parents=True, exist_ok=True)

print('✅ Структура папок создана')

In [ ]:
# ── ЯЧЕЙКА 6: Скачать фото (только нужные классы) ───────────
import random

df = pd.read_csv('fitzpatrick17k.csv')
df['our_class'] = df['label'].str.lower().map(CLASS_MAP)
df_filtered = df[df['our_class'].notna()].copy()

print(f'Отфильтровано: {len(df_filtered)} фото')
print(df_filtered['our_class'].value_counts())

MAX_PER_CLASS = 500  # ограничение для быстрого старта
failed = 0

for cls_name, group in df_filtered.groupby('our_class'):
    sample = group.sample(min(MAX_PER_CLASS, len(group)), random_state=42)
    split_idx = int(len(sample) * 0.8)
    
    for i, (_, row) in enumerate(tqdm(sample.iterrows(), desc=cls_name, total=len(sample))):
        split = 'train' if i < split_idx else 'val'
        save_path = f'/content/dataset/{split}/{cls_name}/{i:05d}.jpg'
        if os.path.exists(save_path):
            continue
        try:
            resp = requests.get(row['url'], timeout=10)
            img = Image.open(BytesIO(resp.content)).convert('RGB')
            img.save(save_path, 'JPEG')
        except Exception:
            failed += 1

print(f'\n✅ Загрузка завершена. Ошибок: {failed}')

In [ ]:
# ── ЯЧЕЙКА 7: Загрузить train.py ────────────────────────────
# Скачать наш train.py (или загрузить вручную)
from google.colab import files
# files.upload()  # загрузите train.py вручную

# ИЛИ скопировать из Drive
# !cp /content/drive/MyDrive/skin_ai/train.py /content/train.py

print('Файлы в /content:')
!ls -la /content/*.py 2>/dev/null || print('train.py не найден — загрузите вручную')

In [ ]:
# ── ЯЧЕЙКА 8: ЗАПУСК ОБУЧЕНИЯ ───────────────────────────────
!python train.py \
    --data_dir /content/dataset \
    --output_dir /content/output \
    --model efficientnet_b3 \
    --epochs 20 \
    --batch_size 32 \
    --lr 1e-4 \
    --img_size 300 \
    --num_workers 2

In [ ]:
# ── ЯЧЕЙКА 9: Сохранить результаты в Google Drive ───────────
import shutil
shutil.copytree('/content/output', f'{DRIVE_PATH}/output', dirs_exist_ok=True)
print('✅ Веса сохранены в Google Drive!')
print(f'📁 {DRIVE_PATH}/output/best_model.pth')

In [ ]:
# ── ЯЧЕЙКА 10: Тест на одном фото ───────────────────────────
import torch
import json
from torchvision import transforms, models
from PIL import Image
import torch.nn as nn

# Загрузка
checkpoint = torch.load('/content/output/best_model.pth', map_location='cpu')
class_map = json.load(open('/content/output/class_map.json'))
num_classes = len(class_map)

model = models.efficientnet_b3(weights=None)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(in_features, num_classes))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

def predict(image_path):
    tf = transforms.Compose([
        transforms.Resize((300, 300)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    img = Image.open(image_path).convert('RGB')
    x = tf(img).unsqueeze(0)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    top5 = probs.topk(min(5, num_classes))
    print('\n🔬 Результат:')
    for prob, idx in zip(top5.values, top5.indices):
        print(f'  {class_map[str(idx.item())]:30s} {prob.item()*100:.1f}%')

# Загрузите тестовое фото
# from google.colab import files
# uploaded = files.upload()
# predict(list(uploaded.keys())[0])

print('✅ Модель загружена и готова к инференсу')
print(f'Точность на валидации: {checkpoint["val_acc"]:.4f}')